# PyTorch basics

This notebook builds the foundations used by TerraClass. You will create tensors, move them between CPU and GPU, use automatic differentiation, define a neural network, and train it on a small synthetic classification problem.

Run the cells in order. Change values and rerun cells whenever you are curious.

## 1. Import PyTorch and select a device

A device is where tensor operations run. TerraClass selects CUDA when an NVIDIA GPU and a CUDA-enabled PyTorch build are available.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Selected device: {device}")

## 2. Create and inspect tensors

A tensor is a multidimensional array. Unlike a plain Python list, it has a fixed data type, a shape, and a device.

In [ ]:
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
zeros = torch.zeros(2, 3)
random_values = torch.randn(2, 3)

print("vector:", vector)
print("matrix:\n", matrix)
print("shape:", matrix.shape)
print("dtype:", matrix.dtype)
print("device:", matrix.device)
print("random values:\n", random_values)

Image batches in TerraClass use the layout `[batch, channels, height, width]`. The next tensor represents eight RGB images that are 64 pixels high and wide.

In [ ]:
images = torch.randn(8, 3, 64, 64)
print("batch:", images.shape[0])
print("channels:", images.shape[1])
print("height and width:", images.shape[2:])
print("number of values:", images.numel())

## 3. Index and reshape tensors

Indexing selects values. Reshaping changes how the same values are organized without changing their total count.

In [ ]:
numbers = torch.arange(12)
grid = numbers.reshape(3, 4)

print("grid:\n", grid)
print("first row:", grid[0])
print("last column:", grid[:, -1])
print("values greater than 5:", grid[grid > 5])
print("flattened again:", grid.flatten())

In [ ]:
one_image = torch.randn(3, 64, 64)
one_image_batch = one_image.unsqueeze(0)

print("single image shape:", one_image.shape)
print("after adding a batch dimension:", one_image_batch.shape)

`unsqueeze(0)` is used during inference because a model expects a batch, even when predicting one image.

## 4. Tensor operations and broadcasting

Most PyTorch operations run element by element. Broadcasting allows compatible smaller tensors to participate without manually copying their values.

In [ ]:
features = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
channel_offset = torch.tensor([0.1, 0.2, 0.3])

print("addition with broadcasting:\n", features + channel_offset)
print("column means:", features.mean(dim=0))
print("row sums:", features.sum(dim=1))

In [ ]:
inputs = torch.tensor([[1.0, 2.0]])       # shape: [1, 2]
weights = torch.tensor([[0.5, -1.0],      # shape: [3, 2]
                        [1.5, 0.0],
                        [0.2, 2.0]])
outputs = inputs @ weights.T              # matrix multiplication

print("output shape:", outputs.shape)
print("outputs:", outputs)

A linear neural-network layer performs this type of matrix multiplication and adds a learned bias.

## 5. Move tensors to the GPU

A model and its input tensors must be on the same device. Calling `.to(device)` returns a tensor on the selected device.

In [ ]:
cpu_tensor = torch.randn(3, 3)
device_tensor = cpu_tensor.to(device)

print("before:", cpu_tensor.device)
print("after:", device_tensor.device)
print("back on CPU:", device_tensor.cpu().device)

## 6. Automatic differentiation

Training requires the derivative of the loss with respect to every model parameter. PyTorch records tensor operations and computes these derivatives with `backward()`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x + 1
y.backward()

print("y:", y.item())
print("dy/dx calculated by PyTorch:", x.grad.item())
print("Expected derivative at x=2: 2*x + 3 =", 2 * x.item() + 3)

Gradients accumulate by default. Training loops therefore call `optimizer.zero_grad()` before each new backward pass.

## 7. Build a neural network with `nn.Module`

The network below accepts two features and returns two logits: one score for each class. ReLU allows it to learn a nonlinear decision boundary.

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, features):
        return self.network(features)


model = TinyClassifier().to(device)
print(model)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
example_batch = torch.randn(4, 2, device=device)
logits = model(example_batch)
probabilities = logits.softmax(dim=1)

print("logit shape:", logits.shape)
print("first example logits:", logits[0].detach().cpu())
print("first example probabilities:", probabilities[0].detach().cpu())
print("probabilities sum to:", probabilities[0].sum().item())

The model returns logits rather than probabilities. `CrossEntropyLoss` expects raw logits and applies the necessary log-softmax internally.

## 8. Create a dataset and data loader

This synthetic problem labels points inside a circle as class 0 and points outside it as class 1. The nonlinear boundary gives the small neural network something meaningful to learn.

In [ ]:
generator = torch.Generator().manual_seed(42)
all_features = torch.randn(1_200, 2, generator=generator)
distance_squared = all_features[:, 0] ** 2 + all_features[:, 1] ** 2
all_labels = (distance_squared > 1.0).long()

train_features, test_features = all_features[:1_000], all_features[1_000:]
train_labels, test_labels = all_labels[:1_000], all_labels[1_000:]

train_dataset = TensorDataset(train_features, train_labels)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

batch_features, batch_labels = next(iter(train_loader))
print("number of training examples:", len(train_dataset))
print("feature batch shape:", batch_features.shape)
print("label batch shape:", batch_labels.shape)

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(
    train_features[:, 0],
    train_features[:, 1],
    c=train_labels,
    cmap="coolwarm",
    alpha=0.6,
    s=16,
)
plt.title("Synthetic two-class dataset")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.show()

## 9. Train the model

Every batch follows the same five operations used by TerraClass: clear gradients, forward pass, calculate loss, backward pass, and update parameters.

In [ ]:
torch.manual_seed(42)
model = TinyClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
loss_history = []

for epoch in range(30):
    model.train()
    total_loss = 0.0

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_features)
        loss = loss_function(logits, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_features.size(0)

    average_loss = total_loss / len(train_dataset)
    loss_history.append(average_loss)
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:02d} | loss: {average_loss:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history)
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.show()

## 10. Evaluate on unseen examples

`model.eval()` switches layers to evaluation behavior. `torch.no_grad()` avoids building a gradient graph because evaluation does not update weights.

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(test_features.to(device))
    test_predictions = test_logits.argmax(dim=1).cpu()

test_accuracy = (test_predictions == test_labels).float().mean().item()
print(f"Test accuracy: {test_accuracy:.1%}")

for index in range(5):
    print(
        f"point={test_features[index].tolist()}, "
        f"actual={test_labels[index].item()}, "
        f"predicted={test_predictions[index].item()}"
    )

## How this maps to TerraClass

| Basics notebook | TerraClass |
| --- | --- |
| `TensorDataset` | TorchVision `EuroSAT` dataset |
| Two numeric features | Three-channel satellite image |
| `TinyClassifier` | `SimpleCNN` or ResNet18 |
| Two output logits | Ten land-use logits |
| Training loop in one cell | `run_epoch` in `engine.py` |
| Accuracy | Accuracy, macro F1, and confusion matrix |

The mechanics are the same; TerraClass uses richer data, convolutional layers, validation, checkpointing, and more complete evaluation.

## Exercises

1. Change the first tensor's dtype to `torch.float64` and inspect it.
2. Change the training batch size from 64 to 16. What happens to the number of optimizer steps?
3. Remove the ReLU layer. Can a linear model learn the circular boundary?
4. Change the hidden layer from 16 to 4 or 64 units and compare accuracy.
5. Reduce training to one epoch, then increase it to 50. Compare loss and test accuracy.
6. Print the gradient norm of the first linear layer immediately after `loss.backward()`.

Next, continue with `01_explore_eurosat.ipynb` to apply these ideas to real satellite images.